# Phase B-0: Volume-Feature Sanity Check (Path B precursor)

**Date:** 2026-04-18.
**Question:** Does adding a volume-matching feature to `combined_score` move phase-1 MAE on either the cohort or the volume-outlier subsets (Q1 low-vol, Q4 high-vol, 5 h/m targets)?

**Motivation:** the scaling diagnostic showed phase 1 under-predicts high-vol movies because `combined_score`'s gap + Jaccard signals don't capture expected review-arrival rate. Volume matching (target's observed rate vs candidate's aligned-window rate) is the most direct fix.

**Design:**

```
combined_v2_score = w_gap × gap_score + w_jac × jaccard + w_vol × vol_score

  gap_score   = exp(−|target_gap − cand_gap| / 8)
  jaccard     = overlap(target_critics, cand_critics_in_aligned_window)
  vol_score   = exp(−|target_rate − cand_rate| / sigma_vol)
    where rate = critics / duration_days, both measured on windows of duration `target_window_days`
```

Sweep `w_vol ∈ {0, 0.1, 0.2, 0.33, 0.5}`. At each w_vol, set `w_gap = w_jac = (1 − w_vol) / 2`. Sigma_vol tuned to cohort rate IQR (below).

**Gate to full Path B:** if w_vol > 0 improves high-vol MAE by ≥10% without cohort regression, it's a cheap win worth shipping; full Path B might still help marginally but isn't urgent. If w_vol moves nothing, the volume signal requires metadata to be useful — full Path B (with TMDb features) is the right investment.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import time

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_selector,
    rate_matched_selector,
    critics_in_window,
    snapshot_state, close_day_count,
    build_critic_profiles, build_kde_lambda_model_capped,
    predict_window_custom,
    passes_skip_rules_for_snap,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SNAP = 3.0

CACHE = CACHE_DIR / 'phase_b0_volume.pkl'
print('Ready.')

## Cohort rate distribution at T-3d (to pick σ_vol)

In [ ]:
rates = []
for target in close_date_map:
    target_gap = gap_for_slug(target)
    if target_gap is None:
        continue
    target_close = close_date_map[target]
    snap_time = target_close - pd.Timedelta(days=SNAP)
    state = snapshot_state(target, snap_time)
    passed, _ = passes_skip_rules_for_snap(state, SNAP)
    if not passed:
        continue
    target_window_days = state['first_review_dbc'] - SNAP
    if target_window_days <= 0:
        continue
    rate = state['observed_count'] / target_window_days
    rates.append({'target': target, 'rate': rate, 'window_days': target_window_days,
                  'n_obs': state['observed_count']})

rates_df = pd.DataFrame(rates)
print(f'Targets usable at T-3d: {len(rates_df)}')
print()
print('Rate (critics/day) distribution:')
print(rates_df['rate'].describe([.1, .25, .5, .75, .9]).round(2).to_string())
print()
iqr = rates_df['rate'].quantile(0.75) - rates_df['rate'].quantile(0.25)
sigma_vol_choice = round(iqr / 2, 1)
print(f'IQR = {iqr:.2f}  →  sigma_vol pick = IQR/2 = {sigma_vol_choice}')

In [ ]:
# Fix sigma_vol based on diagnostic. IQR/2 = 17.8 from the cohort rate distribution above.
SIGMA_VOL = 17.8
print(f'Using SIGMA_VOL = {SIGMA_VOL} (from cohort rate IQR/2)')


## Time a few builds (Phase B.3 compute calibration)

In [ ]:
# Time n=19 and n=21 builds to verify ~500ms estimate
sample_slug = 'lilo_and_stitch_2025'
sample_gap = gap_for_slug(sample_slug)
sample_close = close_date_map[sample_slug]
sample_snap = sample_close - pd.Timedelta(days=SNAP)
sample_state = snapshot_state(sample_slug, sample_snap)
sample_window = sample_state['first_review_dbc'] - SNAP

# Get a 20-slug training set
training_20, _ = combined_score_selector(
    sample_slug, sample_gap, sample_state['observed_critics'], sample_window,
    k=20, alpha=0.5, sigma_gap=8.0,
)

import numpy as np
for n in [19, 20, 21]:
    subset = training_20[:n] if n <= 20 else training_20 + ['despicable_me_4']  # add one
    t0 = time.time()
    for _ in range(3):
        profiles = build_critic_profiles(reviews, close_date_map, subset, verbose=False)
        model = build_kde_lambda_model_capped(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)
    dt = (time.time() - t0) / 3
    print(f'  n={n}: {dt*1000:.0f} ms per build')

## LOO sweep at T-3d

In [ ]:
W_VOL_GRID = [0.0, 0.1, 0.2, 0.33, 0.5]

# Cache keyed by sigma_vol so we can compare runs
CACHE = CACHE_DIR / f'phase_b0_volume_sigma{SIGMA_VOL:g}.pkl'

def run_sweep(force=False):
    if CACHE.exists() and not force:
        return pd.read_pickle(CACHE)

    rows = []
    total = len(close_date_map)
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
        snap_time = target_close - pd.Timedelta(days=SNAP)
        state = snapshot_state(target, snap_time)
        passed, _ = passes_skip_rules_for_snap(state, SNAP)
        if not passed:
            continue

        target_window_days = state['first_review_dbc'] - SNAP
        if target_window_days <= 0:
            continue
        target_critics = state['observed_critics']
        target_rate = state['observed_count'] / target_window_days

        # Ground truth
        movie_reviews = reviews[reviews['movie_slug'] == target].copy()
        movie_reviews['dbc'] = (target_close - movie_reviews['estimated_timestamp']).dt.total_seconds() / 86400
        actual_p1 = int(((movie_reviews['dbc'] > midnight_utc_dbc) & (movie_reviews['dbc'] <= SNAP)).sum())

        for w_vol in W_VOL_GRID:
            w_other = (1 - w_vol) / 2
            if w_vol == 0.0:
                # identical to combined_score(a=0.5)
                training, _ = combined_score_selector(
                    target, target_gap, target_critics, target_window_days,
                    k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
                )
            else:
                training, _ = rate_matched_selector(
                    target, target_gap, target_critics, target_rate, target_window_days,
                    k=SHIP_N_TRAINING,
                    w_gap=w_other, w_jac=w_other, w_vol=w_vol,
                    sigma_gap=SHIP_SIGMA_GAP, sigma_vol=SIGMA_VOL,
                )
            if len(training) < 5:
                continue

            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            model = build_kde_lambda_model_capped(
                profiles,
                bandwidth_floor=SHIP_BANDWIDTH_FLOOR,
                bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
            )
            phase1 = predict_window_custom(
                model, dbc_from=SNAP, dbc_to=midnight_utc_dbc,
                observed_critics=target_critics,
                observed_count=state['observed_count'],
                first_review_dbc=state['first_review_dbc'],
            )
            rows.append({
                'target': target, 'target_gap': target_gap,
                'observed_count': state['observed_count'],
                'target_rate': target_rate,
                'w_vol': w_vol,
                'phase1_pred': float(phase1),
                'actual_phase1': actual_p1,
                'err': float(phase1) - actual_p1,
                'abs_err': abs(float(phase1) - actual_p1),
            })
        if (i + 1) % 30 == 0:
            print(f'  {i+1}/{total} targets')

    df = pd.DataFrame(rows)
    df.to_pickle(CACHE)
    print(f'Cached {len(df)} rows to {CACHE.name}')
    return df

results = run_sweep()


## MAE by (w_vol, stratum)

In [ ]:
def mae_by_w(df, label):
    print(f'{label}:')
    for w in W_VOL_GRID:
        sub = df[df['w_vol'] == w]
        if not len(sub):
            continue
        print(f'  w_vol={w:.2f}  n={len(sub):3d}  MAE={sub["abs_err"].mean():6.2f}  '
              f'mean_err={sub["err"].mean():+6.2f}  median_err={sub["err"].median():+6.2f}')
    print()

# Full cohort
mae_by_w(results, 'Full cohort')

# Stratify by actual_phase1 quartile at w_vol=0 (control)
ctrl = results[results['w_vol'] == 0.0].copy()
ctrl['q_actual'] = pd.qcut(ctrl['actual_phase1'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
quartile_map = dict(zip(ctrl['target'], ctrl['q_actual']))
results['q_actual'] = results['target'].map(quartile_map)

for q in ['Q1','Q2','Q3','Q4']:
    sub = results[results['q_actual'] == q]
    if not len(sub):
        continue
    actual_range = sub['actual_phase1'].min(), sub['actual_phase1'].max()
    mae_by_w(sub, f'{q} (actual_phase1 in [{actual_range[0]}, {actual_range[1]}])')

## H/m subset (the 5 under-predicted live-tracked movies)

In [ ]:
HM_TARGETS = ['the_drama', 'the_super_mario_galaxy_movie',
              'forbidden_fruits_2026', 'they_will_kill_you', 'you_me_and_tuscany']
hm = results[results['target'].isin(HM_TARGETS)].copy()
print('H/m targets by w_vol:')
pivot = hm.pivot_table(index='target', columns='w_vol', values=['phase1_pred', 'err', 'abs_err'])
print(pivot.to_string(float_format='%.2f'))
print()
print('H/m aggregate:')
for w in W_VOL_GRID:
    sub = hm[hm['w_vol'] == w]
    if not len(sub):
        continue
    print(f'  w_vol={w:.2f}  MAE={sub["abs_err"].mean():6.2f}  mean_err={sub["err"].mean():+6.2f}')

## Decision

Read the cohort and Q4/h-m rows across w_vol values:
- If Q4 MAE drops meaningfully with w_vol > 0 without hurting Q1-Q3: volume signal exists, proceed to full Path B (metadata + learned weights) with higher confidence.
- If all strata move together (same direction, same magnitude): volume feature is redundant with gap+Jaccard. Full Path B needs metadata features to contribute anything new.
- If things get worse: the rate-matching signal is noisy at n=20 training. Consider alternative formulations (e.g., rate tertile matching, log-rate).